In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
CHART_DIR = 'charts'
import os; os.makedirs(CHART_DIR, exist_ok=True)

print("✅ Libraries loaded successfully")
print(f"pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")

## 1. Data Loading & Overview

In [ ]:
nav   = pd.read_csv('02_nav_history.csv',        parse_dates=['date'])
fm    = pd.read_csv('01_fund_master.csv')
aum   = pd.read_csv('03_aum_by_fund_house.csv')
sip   = pd.read_csv('04_monthly_sip_inflows.csv', parse_dates=['month'])
cat   = pd.read_csv('05_category_inflows.csv',    parse_dates=['month'])
folio = pd.read_csv('06_industry_folio_count.csv',parse_dates=['month'])
perf  = pd.read_csv('07_scheme_performance.csv')
tx    = pd.read_csv('08_investor_transactions.csv')
hold  = pd.read_csv('09_portfolio_holdings.csv')
bench = pd.read_csv('10_benchmark_indices.csv',   parse_dates=['date'])

datasets = {
    'NAV History': nav, 'Fund Master': fm, 'AUM by Fund House': aum,
    'Monthly SIP': sip, 'Category Inflows': cat, 'Folio Count': folio,
    'Scheme Performance': perf, 'Investor Transactions': tx,
    'Portfolio Holdings': hold, 'Benchmark Indices': bench
}
for name, df in datasets.items():
    print(f"  {name:25s}: {df.shape[0]:>6,} rows × {df.shape[1]:>2} cols")

In [ ]:
print("=== FUND MASTER ===")
print(fm[['fund_house','category','sub_category']].value_counts().head(15))
print("\n=== NAV DATE RANGE ===")
print(f"From: {nav['date'].min().date()}  To: {nav['date'].max().date()}")
print(f"Schemes: {nav['amfi_code'].nunique()}")
print("\n=== TRANSACTION TYPES ===")
print(tx['transaction_type'].value_counts())

## 2. NAV Trend Analysis — All 40 Schemes (2022–2026)

All 40 scheme NAVs are indexed to 100 at Jan 2022. The **green band (2023)** marks the bull run 
driven by FII inflows and domestic recovery. The **red band (2024 Q1–Q4)** marks the correction 
period triggered by global rate concerns and sectoral rotation.

In [ ]:
nav_m = nav.merge(fm[['amfi_code','scheme_name','category']], on='amfi_code', how='left')
nav_m = nav_m.sort_values(['amfi_code','date']).reset_index(drop=True)

nav_piv = nav_m.pivot_table(index='date', columns='amfi_code', values='nav')
nav_norm = (nav_piv / nav_piv.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(16, 8))
palette = plt.cm.tab20(np.linspace(0, 1, nav_norm.shape[1]))
for i, col in enumerate(nav_norm.columns):
    ax.plot(nav_norm.index, nav_norm[col], lw=0.8, alpha=0.55, color=palette[i])

ax.axvspan(pd.Timestamp('2023-01-01'), pd.Timestamp('2023-12-31'),
           alpha=0.12, color='green', label='2023 Bull Run')
ax.axvspan(pd.Timestamp('2024-03-01'), pd.Timestamp('2024-12-31'),
           alpha=0.12, color='red', label='2024 Market Correction')
ax.axhline(100, lw=1, ls='--', color='grey', alpha=0.6)
ax.set_title('NAV Trend – All 40 Schemes (Indexed to 100, Jan 2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Indexed NAV')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/01_nav_trend.png', dpi=150)
plt.show()
print("✅ Chart saved: 01_nav_trend.png")

## 3. AUM Growth by Fund House (2022–2025)

SBI Mutual Fund maintains clear leadership throughout the period, reaching **₹12.5L Cr** in AUM by 2025. 
ICICI Prudential and HDFC follow as strong second-tier players.

In [ ]:
aum['year'] = pd.to_datetime(aum['date']).dt.year
aum_yr = aum.groupby(['fund_house','year'])['aum_lakh_crore'].max().reset_index()
aum_pivot = aum_yr.pivot(index='fund_house', columns='year', values='aum_lakh_crore').fillna(0)

fig, ax = plt.subplots(figsize=(15, 7))
x = np.arange(len(aum_pivot.index))
w = 0.18
years = [c for c in [2022, 2023, 2024, 2025] if c in aum_pivot.columns]
colors = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F']
for i, (yr, col) in enumerate(zip(years, colors)):
    ax.bar(x + i * w, aum_pivot[yr], w, label=str(yr), color=col, alpha=0.85)

sbi_idx = list(aum_pivot.index).index('SBI Mutual Fund')
max_val = aum_pivot.loc['SBI Mutual Fund', years[-1]]
ax.annotate('SBI ₹12.5L Cr\nDominance',
            xy=(sbi_idx + w, max_val),
            xytext=(sbi_idx + 1.6, max_val + 0.4),
            arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5),
            fontsize=9, color='darkred', fontweight='bold')

ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(aum_pivot.index, rotation=35, ha='right', fontsize=9)
ax.set_title('AUM Growth by Fund House (2022–2025)', fontsize=14, fontweight='bold')
ax.set_ylabel('AUM (₹ Lakh Crore)'); ax.legend(title='Year')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/02_aum_growth.png', dpi=150)
plt.show()
print("✅ Chart saved: 02_aum_growth.png")

## 4. SIP Inflow Time-Series (Jan 2022 – Dec 2025)

Monthly SIP inflows grew from ~₹11,500 Cr in Jan 2022 to an all-time high of **₹31,002 Cr in Dec 2025**, 
reflecting growing retail investor participation and financial literacy.

In [ ]:
sip_s = sip.sort_values('month')

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(sip_s['month'], sip_s['sip_inflow_crore'], alpha=0.3, color='#2196F3')
ax.plot(sip_s['month'], sip_s['sip_inflow_crore'], color='#1565C0', lw=2.5, marker='o', ms=3)

ath = sip_s.loc[sip_s['sip_inflow_crore'].idxmax()]
ax.annotate(f"ATH ₹31,002 Cr\n(Dec 2025)",
            xy=(ath['month'], ath['sip_inflow_crore']),
            xytext=(pd.Timestamp('2024-01-01'), ath['sip_inflow_crore'] - 5000),
            arrowprops=dict(arrowstyle='->', color='crimson', lw=1.5),
            fontsize=10, color='crimson', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='crimson'))

ax.set_title('Monthly SIP Inflows (Jan 2022 – Dec 2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('SIP Inflow (₹ Crore)')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/03_sip_inflows.png', dpi=150)
plt.show()
print("✅ Chart saved: 03_sip_inflows.png")

## 5. Category Inflow Heatmap

Months on X-axis, fund categories on Y-axis. Equity-oriented categories (Small Cap, Mid Cap) 
show the most intense inflows during 2024–2025 bull phases.

In [ ]:
cat_pivot = cat.pivot_table(index='category', columns='month', values='net_inflow_crore', aggfunc='sum')
cat_pivot.columns = [c.strftime('%b\n%Y') for c in cat_pivot.columns]

fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(cat_pivot, ax=ax, cmap='RdYlGn', linewidths=0.3,
            annot=True, fmt='.0f', cbar_kws={'label': 'Net Inflow (₹ Crore)'},
            annot_kws={'size': 7})
ax.set_title('Category-wise Net Inflow Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Fund Category')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/04_category_heatmap.png', dpi=150)
plt.show()
print("✅ Chart saved: 04_category_heatmap.png")

## 6. Investor Demographics

The **26–35 age group** dominates with ~41% of investors. Males account for ~66% of transactions. 
SIP amounts are broadly similar across age groups but peak slightly in the 36–45 cohort.

In [ ]:
age_counts    = tx['age_group'].value_counts()
sip_tx        = tx[tx['transaction_type'] == 'SIP']
gender_counts = tx['gender'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
colors_pie = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
axes[0].pie(age_counts, labels=age_counts.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=140,
            wedgeprops=dict(edgecolor='white', lw=1.5))
axes[0].set_title('Age Group Distribution', fontweight='bold')

age_order = ['18-25', '26-35', '36-45', '46-55', '56+']
sns.boxplot(data=sip_tx, x='age_group', y='amount_inr', order=age_order,
            palette='pastel', ax=axes[1], showfliers=False)
axes[1].set_title('SIP Amount by Age Group', fontweight='bold')
axes[1].set_xlabel('Age Group'); axes[1].set_ylabel('SIP Amount (₹)')
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))

axes[2].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#74B9FF', '#FD79A8'], startangle=90,
            wedgeprops=dict(edgecolor='white', lw=1.5))
axes[2].set_title('Gender Split', fontweight='bold')

plt.suptitle('Investor Demographics', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/05_investor_demographics.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved: 05_investor_demographics.png")

## 7. Geographic Distribution

Punjab and Tamil Nadu lead in SIP investment amounts. T30 cities account for ~66% of transactions, 
reflecting urban financial inclusion, though B30 participation is growing.

In [ ]:
state_sip  = tx.groupby('state')['amount_inr'].sum().sort_values(ascending=True).tail(15)
tier_counts = tx['city_tier'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors_bar = plt.cm.Blues(np.linspace(0.4, 0.9, 15))
axes[0].barh(state_sip.index, state_sip.values / 1e7, color=colors_bar)
axes[0].set_xlabel('SIP Amount (₹ Crore)')
axes[0].set_title('Top 15 States by SIP Amount', fontweight='bold')

axes[1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%',
            colors=['#0984E3', '#FDCB6E'], startangle=90,
            wedgeprops=dict(edgecolor='white', lw=1.5))
axes[1].set_title('T30 vs B30 City Tier Split', fontweight='bold')
plt.suptitle('Geographic Distribution of SIP Investments', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/06_geographic_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved: 06_geographic_distribution.png")

## 8. Folio Count Growth (Jan 2022 – Dec 2025)

Industry folios doubled from **13.26 Cr** to **26.12 Cr** in 4 years, reflecting the 
democratisation of mutual fund investing driven by digital platforms and SIP culture.

In [ ]:
folio_s = folio.sort_values('month')

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(folio_s['month'], folio_s['total_folios_crore'], alpha=0.2, color='#6C5CE7')
ax.plot(folio_s['month'], folio_s['total_folios_crore'], color='#6C5CE7', lw=2.5, marker='o', ms=5)

for dt, lbl in [('2022-01', '13.26 Cr'), ('2025-12', '26.12 Cr')]:
    row = folio_s.iloc[(folio_s['month'] - pd.Timestamp(dt)).abs().argsort().iloc[0]]
    ax.annotate(lbl, xy=(row['month'], row['total_folios_crore']),
                xytext=(row['month'], row['total_folios_crore'] + 0.8),
                ha='center', fontsize=9, fontweight='bold', color='#6C5CE7',
                arrowprops=dict(arrowstyle='->', color='#6C5CE7', lw=1.2))

ax.set_title('Industry Folio Count Growth (Jan 2022 – Dec 2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Total Folios (Crore)')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/07_folio_count.png', dpi=150)
plt.show()
print("✅ Chart saved: 07_folio_count.png")

## 9. NAV Return Correlation Matrix (10 Selected Funds)

Pairwise Pearson correlation of daily returns computed for 10 funds. High intra-equity correlation 
(>0.7) is expected; cross-category (equity vs debt) correlations are near zero or negative — 
confirming diversification benefits.

In [ ]:
sel = nav['amfi_code'].unique()[:10]
nav_sel = nav[nav['amfi_code'].isin(sel)].copy()
nav_rpiv = nav_sel.pivot_table(index='date', columns='amfi_code', values='nav')
rets = nav_rpiv.pct_change().dropna()

code2name = fm.set_index('amfi_code')['scheme_name'].str.split(' - ').str[0]
rets.columns = [str(code2name.get(c, c))[:20] for c in rets.columns]

fig, ax = plt.subplots(figsize=(12, 10))
corr = rets.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, mask=mask, linewidths=0.5,
            cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Pairwise Return Correlation – 10 Selected Funds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/08_return_correlation.png', dpi=150)
plt.show()
print("✅ Chart saved: 08_return_correlation.png")

## 10. Sector Allocation Donut — Equity Funds

Aggregated sector weights from portfolio_holdings.csv across all equity fund holdings. 
**Banking & Financial Services** dominates, followed by Technology and Consumer Goods — 
consistent with NIFTY 50 composition.

In [ ]:
eq_codes = fm[fm['category'] == 'Equity']['amfi_code'].tolist()
hold_eq  = hold[hold['amfi_code'].isin(eq_codes)]
sector_wt = hold_eq.groupby('sector')['weight_pct'].sum().sort_values(ascending=False)

print("Top sectors:")
print(sector_wt.to_string())

fig, ax = plt.subplots(figsize=(12, 9))
colors_d = plt.cm.Set3(np.linspace(0, 1, len(sector_wt)))
ax.pie(sector_wt, labels=sector_wt.index, autopct='%1.1f%%',
       colors=colors_d, startangle=90, pctdistance=0.82,
       wedgeprops=dict(width=0.55, edgecolor='white', lw=1.5))
ax.set_title('Sector Allocation – Equity Funds\n(Aggregate Portfolio Holdings)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/09_sector_donut.png', dpi=150)
plt.show()
print("✅ Chart saved: 09_sector_donut.png")

## 11. Additional Charts — Risk-Return, Accounts, AUM, Benchmarks, Expense, Folio Mix

In [ ]:
# Chart 10: Risk-Return Scatter
fig, ax = plt.subplots(figsize=(12, 7))
cat_colors = {'Equity':'#0984E3', 'Debt':'#74B9FF', 'Hybrid':'#FD79A8', 'Solution Oriented':'#6C5CE7'}
for cat_name, grp in perf.groupby('category'):
    c = cat_colors.get(cat_name, 'grey')
    ax.scatter(grp['std_dev_ann_pct'], grp['return_3yr_pct'], label=cat_name,
               color=c, alpha=0.85, s=80, edgecolors='white', lw=0.5)
ax.axhline(perf['return_3yr_pct'].mean(), ls='--', color='grey', alpha=0.6, label='Avg Return')
ax.set_xlabel('Annualised Std Dev (%)'); ax.set_ylabel('3-Year Return (%)')
ax.set_title('Risk-Return Scatter – All Schemes (3-Year)', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/10_risk_return_scatter.png', dpi=150)
plt.show(); print("✅ 10_risk_return_scatter.png")

In [ ]:
# Chart 11: SIP Inflow vs Active Accounts
fig, ax = plt.subplots(figsize=(13, 5))
ax2 = ax.twinx()
ax.bar(sip_s['month'], sip_s['sip_inflow_crore'], width=25, color='#74B9FF', alpha=0.7, label='SIP Inflow (₹ Cr)')
ax2.plot(sip_s['month'], sip_s['active_sip_accounts_crore'], color='#E17055', lw=2.5, marker='o', ms=4, label='Active Accounts (Cr)')
ax.set_ylabel('Monthly SIP Inflow (₹ Crore)', color='#74B9FF')
ax2.set_ylabel('Active SIP Accounts (Crore)', color='#E17055')
ax.set_title('SIP Inflow vs Active SIP Accounts Growth', fontsize=13, fontweight='bold')
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1+h2, l1+l2, loc='upper left')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/11_sip_accounts.png', dpi=150)
plt.show(); print("✅ 11_sip_accounts.png")

In [ ]:
# Chart 12: Top 10 Funds by AUM
top10 = perf.nlargest(10, 'aum_crore').copy()
top10['label'] = top10['scheme_name'].str.split(' - ').str[0].str[:35]
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top10['label'], top10['aum_crore']/1000,
               color=plt.cm.Blues(np.linspace(0.4,0.9,10)), edgecolor='white')
ax.bar_label(bars, labels=[f'₹{v:.0f}K Cr' for v in top10['aum_crore']/1000], padding=3, fontsize=9)
ax.set_xlabel('AUM (₹ Thousand Crore)')
ax.set_title('Top 10 Mutual Fund Schemes by AUM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/12_top10_aum.png', dpi=150)
plt.show(); print("✅ 12_top10_aum.png")

In [ ]:
# Chart 13: Benchmark Comparison
bench_piv  = bench.pivot_table(index='date', columns='index_name', values='close_value')
bench_norm = (bench_piv / bench_piv.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(14, 6))
cmap = plt.cm.tab10(np.linspace(0, 1, len(bench_norm.columns)))
for col, c in zip(bench_norm.columns, cmap):
    ax.plot(bench_norm.index, bench_norm[col], label=col, lw=2, color=c)
ax.axvspan(pd.Timestamp('2023-01-01'), pd.Timestamp('2023-12-31'), alpha=0.1, color='green')
ax.axvspan(pd.Timestamp('2024-03-01'), pd.Timestamp('2024-12-31'), alpha=0.1, color='red')
ax.axhline(100, ls='--', color='grey', lw=0.8)
ax.set_title('Benchmark Index Comparison (Indexed, 2022–2026)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Indexed Value (Base=100)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/13_benchmark_comparison.png', dpi=150)
plt.show(); print("✅ 13_benchmark_comparison.png")

In [ ]:
# Chart 14: Expense Ratio by Category
fig, ax = plt.subplots(figsize=(12, 6))
cat_ord = perf.groupby('category')['expense_ratio_pct'].median().sort_values(ascending=False).index
sns.violinplot(data=perf, x='category', y='expense_ratio_pct', order=cat_ord, palette='muted', ax=ax, inner='box')
ax.set_title('Expense Ratio Distribution by Fund Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Category'); ax.set_ylabel('Expense Ratio (%)')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/14_expense_ratio.png', dpi=150)
plt.show(); print("✅ 14_expense_ratio.png")

In [ ]:
# Chart 15: Folio Category Mix
folio_s = folio.sort_values('month')
fig, ax = plt.subplots(figsize=(13, 6))
ax.stackplot(folio_s['month'],
             folio_s['equity_folios_crore'], folio_s['debt_folios_crore'],
             folio_s['hybrid_folios_crore'], folio_s['others_folios_crore'],
             labels=['Equity','Debt','Hybrid','Others'],
             colors=['#74B9FF','#FDCB6E','#55EFC4','#FD79A8'], alpha=0.85)
ax.set_title('Folio Category Mix Over Time (Stacked Area)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Folios (Crore)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/15_folio_mix.png', dpi=150)
plt.show(); print("✅ 15_folio_mix.png")

## 12. Key EDA Findings

### Finding 1 — SIP Inflows Tripled Over 4 Years
Monthly SIP inflows grew from ₹11,517 Cr (Jan 2022) to ₹31,002 Cr (Dec 2025), nearly tripling in 
4 years — driven by digital onboarding and financial awareness. *(Ref: Chart 03_sip_inflows.png)*

### Finding 2 — SBI Mutual Fund Leads AUM with ₹12.5L Cr
SBI Mutual Fund surpassed all peers with ₹12.5 Lakh Crore AUM by 2025, more than double its nearest 
competitor (ICICI Prudential at ~₹6.2L Cr). *(Ref: Chart 02_aum_growth.png)*

### Finding 3 — Folio Count Doubled in 4 Years
Total folios grew from 13.26 Crore (Jan 2022) to 26.12 Crore (Dec 2025), reflecting new-to-MF 
investor participation through platforms like Zerodha, Groww, and CAMS. *(Ref: Chart 07_folio_count.png)*

### Finding 4 — 2023 Bull Run Delivered 30–80% NAV Gains
Most equity schemes delivered 30–80% indexed returns during the 2023 green-shaded period, 
outperforming the NIFTY 50 base by significant margins. *(Ref: Chart 01_nav_trend.png)*

### Finding 5 — Small Cap and Mid Cap Categories Led 2024 Inflows
The heatmap shows Deep Green (high inflows) concentrated in Small Cap and Mid Cap categories 
during 2024–2025, signalling retail preference for higher-growth categories. *(Ref: Chart 04_category_heatmap.png)*

### Finding 6 — 26–35 Age Group Drives ~41% of Investor Transactions
The millennial cohort dominates the investor base, underscoring the impact of mobile-first 
investment apps targeting young salaried professionals. *(Ref: Chart 05_investor_demographics.png)*

### Finding 7 — Banking & Financial Services Lead Equity Portfolio Allocation
Aggregating sector weights across all equity fund holdings, Banking (~22%) and NBFC (~10%) 
together form the largest block — consistent with Nifty 50 index composition. *(Ref: Chart 09_sector_donut.png)*

### Finding 8 — High Intra-Equity Return Correlation (>0.75)
All equity scheme pairs exhibit correlation above 0.75, suggesting limited intra-equity 
diversification benefits; debt funds show near-zero or negative correlation with equity. *(Ref: Chart 08_return_correlation.png)*

### Finding 9 — T30 Cities Contribute ~66% of SIP Transactions
Urban (T30) cities dominate investment volumes, though B30 cities (~34%) are growing faster 
as rural digitalisation and Jan Dhan-linked MF schemes take hold. *(Ref: Chart 06_geographic_distribution.png)*

### Finding 10 — Equity Funds Charge the Highest Expense Ratios
Equity funds have the widest expense ratio distribution (0.10%–2.50%), while debt funds are 
more tightly clustered near 0.25%–1.00% — directly impacting long-term real returns. *(Ref: Chart 14_expense_ratio.png)*

In [ ]:
import os
charts = sorted(os.listdir(CHART_DIR))
print(f"\n📊 Total Charts Generated: {len(charts)}")
for c in charts:
    print(f"  ✅ {c}")
print("\n✅ EDA_Analysis.ipynb complete — Day 3 Task Done!")